In [344]:
#import any necessary libraries
import pandas as pd
import random
import ipywidgets as widgets
from IPython.display import display
from IPython.display import Image as NotebookImage
from IPython.display import Image as HTMLImage
from IPython.display import HTML
import requests
import json

In [287]:
# read animal dataset
animals_df = pd.read_csv('data/Animal Characteristics Data.csv')

# read scenario file
from pathlib import Path
scenarios = json.loads(Path('data/scenarios.json').read_text())

In [320]:
#Select an animal (the game is started in here!!!)
def select_animal(change): 
    global animal
    name = text_input.value.strip().capitalize()
    animal = animals_df.loc[animals_df.animal == name]

    with output:
        output.clear_output(wait=True)
        if animal.empty:
            print("Animal not found. Try again!")
        else:
            print("You selected:", animal["animal"].iloc[0])
            #update to update the global animal and change the output?
            start_game(animal)

In [270]:
def display_stats():
    print(f"Animal: {animal["animal"].iloc[0]}")
    print(f"Health: {animal_hp}")
    print(f"Energy: {animal_nrg}")
    print(f"Score: {player_score}")
    print("")

In [302]:
#find the percentile of a stat in a datast
def remap_num(animal, col):
    return animals_df[col].rank(pct=True).loc[animal.index[0]] * 100
    
# calculate game variables based on animal
def set_stats(animal):
    starting_stat = 30
    # variable that cuts the percentile to prevent being too OP
    factor = 4
    
    # health
    animal_hp = round(
        starting_stat
        + (remap_num(animal, "weight_kg") 
        + remap_num(animal, "lifespan_years")
          ) / factor
    )
    
    animal_nrg = round(
        starting_stat 
        + (remap_num(animal, "speed_kmh")
        + (4 * animal["can_swim"].iloc[0])
        + (2 * animal["can_fly"].iloc[0])
        + (2 *animal["has_wings"].iloc[0])
          )/ factor
    )
    return animal_hp, animal_nrg

In [131]:
# unused variables below
# size, lays_eggs, number_of_legs, has_four_legs, has_a_tail, has_spots, color, habitat, has_fur, active_at_night, social, diet

#print stats block
display_stats()

Animal: Bee
Health: 72
Energy: 0
Score: 185


In [351]:
def get_event():
    global curr_event, choices
    curr_event = scenarios[random.choice(list(scenarios.keys()))]
    choices = curr_event["choices"]

#define what happens when a choice is made
def choice_selected(button, choice):
    global animal_hp, animal_nrg, player_score 
    animal_hp += choice["health"]
    animal_nrg += choice["energy"]
    player_score += choice["score"]

    if animal_hp <= 0 or animal_nrg <= 0:
        with output: 
            output.clear_output(wait=True)
            display_stats()
            print("You died!")
            #display picture of animal but change the style so you know somehting happened
            display(HTML(f'<img src="{animal["image_url"].iloc[0]}" width="200" height="200" style="filter: sepia(100%) saturate(300%) hue-rotate(-50deg);" />'))
            #allow the user to enter a new animal and try again if they want 

            
            
    else:
        get_event()
        show_event()

#display the choice buttons and scenatio image
def show_event():
    # print the initial pre-round text layout
    with output:  #needed help

        #clear the prev output 
        output.clear_output(wait=True)
        buttons = []

        #display picture of animal
        display_url(animal["image_url"].iloc[0])
        
        #print the current event
        display_stats()
        print(curr_event["name"])
        print(curr_event["description"])

        
        # if a button is clicked, apply the impact of the choice made
        for choice in choices.values():
            button = widgets.Button(description=choice["description"], layout=widgets.Layout(width='auto'))
            button.on_click(lambda b, c=choice: choice_selected(b, c)) #needed help with "lambda"
            buttons.append(button)

        #display the image
        display_url(curr_event["scenario_image"])
        
        #display updated buttons
        display(widgets.VBox(buttons))

In [349]:
#show image from url
def display_url(url):
    display(HTMLImage(url=url, width=200, height=200))

#I discovered that this was denied by the webiste that has the images... pivoted to HTML display to accomodate
# def display_url(url):
#     response = requests.get(url)
#     img = widgets.Image(
#         value=response.content,
#         format="jpeg",
#         width=200,
#         height=200
#     )
#     display(img)


In [308]:


#CHAT DO NOT WORRY ABOUT THIS YET
#allow a try again button
#visualize the stats better

In [348]:
text_input = widgets.Text(placeholder = 'Enter an animal')
display(text_input)

#original on submit was depricated- error warning reccomended the following logic
text_input.observe(select_animal, names="value")

output = widgets.Output()
display(output)

#reset the stats
animal_hp, animal_nrg = set_stats(animal)
player_score = 0

Text(value='', placeholder='Enter an animal')

Output()

In [339]:
def start_game(animal):
    global animal_hp, animal_nrg, player_score

    #reset the stats
    animal_hp, animal_nrg = set_stats(animal)
    player_score = 0

    #init the scenarios 
    get_event()
    show_event()


    
    



In [350]:
output = widgets.Output()

text_input = widgets.Text(placeholder = 'Enter an animal')
text_input.observe(select_animal, names="value")
#text_input.continuous_update = False

display(text_input)
display(output)

Text(value='', placeholder='Enter an animal')

Output()